In [1]:
%load_ext autoreload
%autoreload 2


## Libraries

# Variable factory

In [2]:
import sys

# Add root to path
sys.path.append("..")


In [3]:
from typing import Literal

from spatialrisk.session import ProjectSession
from spatialrisk.document import GEESpec, CatalogueRecipe
from spatialrisk.variables.models import DataType, RasterType, RasterizationMethod
import ee


## GEE


In [4]:

ee.Authenticate()


True

In [5]:
ee_project = "ee-joseserafini-fao"
ee.Initialize(project=ee_project)


## Set project parameters

In [6]:
project_name = "mtq-refactor"
session = ProjectSession.create(project_name)


In [7]:
# To resume an existing project instead of creating a new one:
# session = ProjectSession.open(project_name)


In [8]:
# Registered variables (empty for a fresh project)
session.list_variables()


project.variables → {'raw': {...}, 'processed': {...}} view


{'raw': {}, 'processed': {}}

In [9]:
# Persistence is explicit now (no auto-save on every mutation).
session.save()


Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json


PosixPath('/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json')

In [10]:
session.snapshot()


Project(project_name='mtq-refactor', years=None, raw_variables={}, processed_variables={}, base_raster=None, models={}, datasets={})

# Create an area of interest

In [11]:
iso_code = "MTQ"

# The AOI recipe produces the project AOI GeoJSON once; everything downstream
# rebuilds ee.Geometry from session AOI in-worker.
session.add_gee_variable(
    GEESpec(
        name="aoi",
        data_type=DataType.vector,
        recipe=CatalogueRecipe(
            source="catalogue",
            catalogue_key="aoi_fao_gaul",
            params={"level": 0, "iso": iso_code},
            export_kind="vector",
            vector_selectors=("gaul0_name", "iso3_code"),
        ),
    )
)
# Materialize the AOI vector (needs GEE credentials), then set the project AOI
# from the downloaded shapefile. Full execution requires real EE auth.
import geopandas as gpd
import json

session.materialize_all(source="raw")          # downloads the 'aoi' vector
aoi_spec = session._collection("raw")["aoi"]
aoi_path = session._collection("raw")[aoi_spec.materialized_key].path
aoi_geojson = json.loads(gpd.read_file(aoi_path).to_json())
session.set_aoi(aoi_geojson["features"][0]["geometry"])


/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data_raw/aoi.shp already exists. Skipping download.


In [12]:
# Local-file AOI alternative:
# from spatialrisk.document import LocalVectorSpec
# session.add_local_vector(LocalVectorSpec(
#     name="aoi", path="/path/to/aoi.shp", rasterization_method="binary"))
# session.materialize_all(source="raw")
# ... then read + session.set_aoi(...) as above

## SubJuridistion

In [13]:
session.add_gee_variable(
    GEESpec(
        name="subj",
        data_type=DataType.raster,
        raster_type=RasterType.categorical,
        recipe=CatalogueRecipe(
            source="catalogue",
            catalogue_key="subj",
            params={"gaul_level": 2, "gaul_column": "gaul2_name"},
            export_kind="raster",
        ),
    )
)


  0%|          |0/4 tiles [00:00<?]

File /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data_raw/subj.tif, downloaded
✓ Added 'subj' to raw variables (key: subj)
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json


/home/jserafini/micromamba/envs/spatial-risk/lib/python3.11/site-packages/geedim/image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'None'.
  return STACClient().get(self.id)


# GEE VARIABLES

## Areas protegidas

In [14]:
session.add_gee_variable(
    GEESpec(
        name="protected_area",
        data_type=DataType.raster,
        raster_type=RasterType.categorical,
        recipe=CatalogueRecipe(
            source="catalogue",
            catalogue_key="protected_area",
            export_kind="raster",
        ),
    )
)


/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data_raw/protected_area.tif already exists. Skipping download.
✓ Added 'protected_area' to raw variables (key: protected_area)
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json


## Altitude

In [15]:
session.add_gee_variable(
    GEESpec(
        name="altitude",
        data_type=DataType.raster,
        raster_type=RasterType.continuous,
        recipe=CatalogueRecipe(
            source="catalogue",
            catalogue_key="altitude",
            export_kind="raster",
        ),
    )
)


/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data_raw/altitude.tif already exists. Skipping download.
✓ Added 'altitude' to raw variables (key: altitude)
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json


## Slope

In [16]:
session.add_gee_variable(
    GEESpec(
        name="slope",
        data_type=DataType.raster,
        raster_type=RasterType.continuous,
        recipe=CatalogueRecipe(
            source="catalogue",
            catalogue_key="slope",
            export_kind="raster",
        ),
    )
)


/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data_raw/slope.tif already exists. Skipping download.
✓ Added 'slope' to raw variables (key: slope)
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json


## Forest layers

In [17]:
forest_source = "gfc"  # "gfc" or "tmf"
tree_cover_threshold = 10  # percent
years = [2015, 2020, 2024]

for year in years:
    session.add_gee_variable(
        GEESpec(
            name="forest_gfc" if forest_source == "gfc" else "forest_tmf",
            data_type=DataType.raster,
            raster_type=RasterType.categorical,
            year=year,
            tags=("forest",),
            recipe=CatalogueRecipe(
                source="catalogue",
                catalogue_key="forest_gfc" if forest_source == "gfc" else "forest_tmf",
                params={
                    "year": year,
                    "forest_source": forest_source,
                    "tree_cover_threshold": tree_cover_threshold,
                },
                export_kind="raster",
            ),
        )
    )


/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data_raw/forest_gfc_2015.tif already exists. Skipping download.
✓ Added 'forest_gfc' to raw variables (key: forest_gfc_2015)
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data_raw/forest_gfc_2020.tif already exists. Skipping download.
✓ Added 'forest_gfc' to raw variables (key: forest_gfc_2020)
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data_raw/forest_gfc_2024.tif already exists. Skipping download.
✓ Added 'forest_gfc' to raw variables (key: forest_gfc_2024)
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_pr

/home/jserafini/micromamba/envs/spatial-risk/lib/python3.11/site-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for UMD/hansen/global_forest_change_2024_v1_12! You are using a deprecated asset.
To ensure continued functionality, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2024_v1_12

  warnings.warn(warning, category=DeprecationWarning)


## Rivers

In [18]:
session.add_gee_variable(
    GEESpec(
        name="rivers",
        data_type=DataType.raster,
        raster_type=RasterType.categorical,
        tags=("river",),
        recipe=CatalogueRecipe(
            source="catalogue",
            catalogue_key="rivers",
            export_kind="raster",
        ),
    )
)


/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data_raw/rivers.tif already exists. Skipping download.
✓ Added 'rivers' to raw variables (key: rivers)
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json


## Roads

In [19]:
session.add_gee_variable(
    GEESpec(
        name="roads",
        data_type=DataType.raster,
        raster_type=RasterType.categorical,
        recipe=CatalogueRecipe(
            source="catalogue",
            catalogue_key="roads",
            export_kind="raster",
        ),
    )
)


/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data_raw/roads.tif already exists. Skipping download.
✓ Added 'roads' to raw variables (key: roads)
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json


## Towns

In [20]:
for year in years:
    session.add_gee_variable(
        GEESpec(
            name="towns",
            data_type=DataType.raster,
            raster_type=RasterType.categorical,
            year=year,
            tags=("town",),
            recipe=CatalogueRecipe(
                source="catalogue",
                catalogue_key="towns",
                params={"year": year},
                export_kind="raster",
            ),
        )
    )


/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data_raw/towns_2015.tif already exists. Skipping download.
✓ Added 'towns' to raw variables (key: towns_2015)
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data_raw/towns_2020.tif already exists. Skipping download.
✓ Added 'towns' to raw variables (key: towns_2020)
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json
/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/data_raw/towns_2024.tif already exists. Skipping download.
✓ Added 'towns' to raw variables (key: towns_2024)
Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json


In [21]:
# Download GEE specs, reproject/match to the base raster, rasterize vectors.
session.process_all()
session.save()


Project saved to: /home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json


PosixPath('/home/jserafini/Desktop/projects/spatial_risk/spatial-risk-module/data/mtq-refactor/mtq-refactor_project.json')

In [22]:
## Consider using Global Human Modification v3
# https://gee-community-catalog.org/projects/ghm/?h=human

## Oxford accessibility to cities 2015
## Oxford/MAP/accessibility_to_cities_2015_v1_0


# Custom variables

### GEEVars

In [23]:
# Ad-hoc user asset (not in the catalogue):
# from spatialrisk.document import AssetRecipe
# session.add_gee_variable(
#     GEESpec(
#         name="varname",
#         data_type=DataType.raster,
#         raster_type=RasterType.continuous,
#         recipe=AssetRecipe(
#             source="asset",
#             asset_id="users/me/my_layer",
#             band="b1",
#             export_kind="raster",
#         ),
#     )
# )


GEEVar(name='varname', data_type='raster', year=None, active=True, tags=[], path=None, default_scale=None, default_crs=None, raster_type=None, rasterization_method=None, post_processing=[])

### LocalVector

In [24]:
# session.add_local_vector(
#     name="centros_poblados",
#     path="/path/to/Centrospoblados.shp",
#     rasterization_method=RasterizationMethod.binary,
# )


### LocalRaster

In [25]:
# session.add_local_raster(
#     name="coca",
#     path="/path/to/DE_COCA.img",
#     raster_type=RasterType.categorical,
# )


In [26]:
# TODO: make an example
